<a href="https://colab.research.google.com/github/maiphuong270704-hash/PHAM-THI-MAI-PHUONG/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1-1

In [ ]:
# =======================================================
# 「說了算！」語音輸入待辦清單 (Google Colab 完整原型 - SQLite 資料庫版)
# 說明：此程式碼必須在 Google Colab 或具備麥克風權限的環境中運行。
# =======================================================

# 1. 環境建置與函式庫匯入
# --- 請確保在 Colab 環境下執行此行以安裝所需套件 ---
!pip install gradio SpeechRecognition python-dateutil

import gradio as gr
import speech_recognition as sr
from dateutil import parser
from datetime import datetime
import uuid
import sqlite3
import os

# 2. 資料庫設定與初始化
DB_NAME = 'todo_tasks.db'

def initialize_db():
    """建立資料庫連線，並確保 'tasks' 資料表存在。"""

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    # 建立資料表，包含 uuid (UUID)、task_content (任務內容)、due_datetime (截止時間)
    c.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            uuid TEXT NOT NULL UNIQUE,
            task_content TEXT NOT NULL,
            due_datetime TEXT NOT NULL,
            status TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()

# 立即執行初始化
initialize_db()
print(f"資料庫 '{DB_NAME}' 初始化完成，確保 'tasks' 資料表已建立。")


# 3. 核心功能函數定義 (資料庫操作)

def parse_date_time(text):
    """嘗試從文字中解析出日期和時間，並處理日期調整邏輯 (S7: 基礎日期時間解析)。"""

    try:
        dt = parser.parse(text, fuzzy=True, dayfirst=True)

        # 簡化處理：如果解析出來的時間是在過去，調整為未來
        now = datetime.now()
        if dt < now:
            if dt.day == now.day and dt.month == now.month and dt.year == now.year:
                dt = dt.replace(day=now.day + 1)
            else:
                dt = dt.replace(year=now.year + 1)

        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except (ValueError, TypeError):
        return datetime.now().strftime('%Y-%m-%d 無具體時間')

def transcribe_audio(audio_file_path):
    """將 Gradio 提供的音訊檔案路徑轉換為文字 (S2: 語音轉文字)。"""

    if not audio_file_path:
        return "錯誤：未接收到音訊檔案，請重新錄音。", ""

    r = sr.Recognizer()
    try:
        with sr.AudioFile(audio_file_path) as source:
            audio = r.record(source)

        # 設定語言為中文 (zh-TW)
        text = r.recognize_google(audio, language="zh-TW")
        return "語音辨識成功。", text

    except sr.UnknownValueError:
        return "語音辨識服務無法理解音訊內容。", "請嘗試更清晰地說話。"
    except sr.RequestError as e:
        return f"錯誤：無法連線至 Google 語音辨識服務; {e}", ""
    except Exception as e:
        return f"發生未知錯誤: {e}", ""

def add_task(transcribed_text):
    """根據辨識出的文字，解析日期時間並新增任務至資料庫。"""
    if not transcribed_text or "請嘗試" in transcribed_text:
        return "任務新增失敗：請提供有效的語音輸入。", format_tasks()

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    try:
        due_date = parse_date_time(transcribed_text)

        # 寫入資料庫 (S6: 持久化儲存)
        c.execute("""
            INSERT INTO tasks (uuid, task_content, due_datetime, status, created_at)
            VALUES (?, ?, ?, ?, ?)
        """, (
            str(uuid.uuid4())[:8],
            transcribed_text,
            due_date,
            '待辦',
            datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        ))
        conn.commit()

        return f"任務新增成功！標題：'{transcribed_text}'，預計時間：{due_date}", format_tasks()

    except Exception as e:
        return f"資料庫新增錯誤: {e}", format_tasks()
    finally:
        conn.close()

def complete_task_by_id(task_id_input):
    """根據 UUID 標記任務為完成 (S5: 任務完成標記)。"""
    task_id_input = task_id_input.strip()
    if not task_id_input:
        return "請輸入有效的任務 ID。", format_tasks()

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    try:
        # 更新資料庫
        c.execute("""
            UPDATE tasks SET status = '完成'
            WHERE uuid = ? AND status = '待辦'
        """, (task_id_input,))

        if c.rowcount > 0:
            conn.commit()
            return f"任務 ID {task_id_input} 已標記為「完成」。", format_tasks()
        else:
            return f"找不到 ID 為 {task_id_input} 的待辦任務或該任務已完成。", format_tasks()

    except Exception as e:
        return f"資料庫更新錯誤: {e}", format_tasks()
    finally:
        conn.close()

def format_tasks():
    """從資料庫讀取任務列表，並轉換為 HTML 表格格式 (S3: 清單查看與排序)。"""

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    # 查詢優化：依照截止時間排序，未完成的在前
    c.execute("""
        SELECT uuid, task_content, due_datetime, status
        FROM tasks
        ORDER BY
            CASE status WHEN '待辦' THEN 0 ELSE 1 END,
            due_datetime ASC
    """)
    tasks_data = c.fetchall()
    conn.close()

    if not tasks_data:
        return "<p>目前沒有任何待辦事項。</p>"

    # 建立 HTML 表格
    html = """
    <style>
        .task-table { width: 100%; border-collapse: collapse; font-family: Arial, sans-serif; }
        .task-table th, .task-table td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        .task-table th { background-color: #f2f2f2; }
        .status-待辦 { color: red; font-weight: bold; }
        .status-完成 { color: green; }
        .task-row-complete { text-decoration: line-through; color: #888; }
    </style>
    <table class="task-table">
        <tr>
            <th>ID (UUID)</th>
            <th>任務內容</th>
            <th>預計時間</th>
            <th>狀態</th>
        </tr>
    """

    for task in tasks_data:
        uuid_val, content, due_date, status = task
        status_class = f"status-{status}"
        row_class = "task-row-complete" if status == '完成' else ""

        html += f"""
        <tr class="{row_class}">
            <td>{uuid_val}</td>
            <td>{content}</td>
            <td>{due_date}</td>
            <td class="{status_class}">{status}</td>
        </tr>
        """

    html += "</table>"
    return html

# 4. Gradio 介面定義與流程串接

with gr.Blocks(title="說了算！語音輸入待辦清單 (SQLite 版)") as demo:
    gr.Markdown("<h1>🗣️ 「說了算！」語音輸入待辦清單 (SQLite 資料庫版)</h1>")
    gr.Markdown("請允許瀏覽器使用麥克風權限。所有任務數據已實現**持久化**，儲存在 Colab 虛擬機中的 `todo_tasks.db` 檔案裡。")

    # 狀態輸出區
    status_output = gr.Textbox(label="系統訊息", interactive=False, value="準備就緒...")

    with gr.Row():
        # 錄音輸入區
        with gr.Column(scale=1):
            gr.Markdown("<h3>步驟一：語音輸入與解析</h3>")
            # 語音輸入元件 (S1: 語音輸入啟動)
            audio_input = gr.Audio(sources=["microphone"], type="filepath", label="請按住或點擊錄音按鈕說出任務")

            transcribe_status = gr.Textbox(label="辨識狀態", interactive=False)
            transcribed_text = gr.Textbox(label="辨識出的文字內容", interactive=False)

            add_btn = gr.Button("解析並新增任務 ➕", variant="primary")

        # 任務完成區
        with gr.Column(scale=1):
            gr.Markdown("<h3>步驟二：標記任務完成</h3>")
            complete_id_input = gr.Textbox(label="輸入要標記為「完成」的任務 ID")
            complete_btn = gr.Button("標記為完成 ✅", variant="secondary")

    gr.Markdown("<h2>📝 待辦清單</h2>")

    # 任務清單顯示區
    task_list_output = gr.HTML(format_tasks(), label="目前待辦清單")

    # ----- 流程串接 -----

    # 1. 錄音後，先執行語音轉文字
    audio_input.change(
        fn=transcribe_audio,
        inputs=[audio_input],
        outputs=[transcribe_status, transcribed_text]
    )

    # 2. 點擊新增任務按鈕，執行解析並新增任務
    add_btn.click(
        fn=add_task,
        inputs=[transcribed_text],
        outputs=[status_output, task_list_output]
    )

    # 3. 點擊標記完成按鈕
    complete_btn.click(
        fn=complete_task_by_id,
        inputs=[complete_id_input],
        outputs=[status_output, task_list_output]
    )

    # 4. 運行 Gradio 介面
    demo.launch(share=True)